# GraphRAG Interactive Visualization with yfiles-jupyter-graphs

This notebook demonstrates how to use GraphRAG's built-in visualization capabilities.

In [ ]:
# Install yfiles-jupyter-graphs if not already installed
%pip install yfiles-jupyter-graphs --quiet

import pandas as pd
from yfiles_jupyter_graphs import GraphWidget
from IPython.display import display
import os

In [ ]:
# Load GraphRAG data
OUTPUT_DIR = "workspace/output"

entities_df = pd.read_parquet(f"{OUTPUT_DIR}/entities.parquet")
relationships_df = pd.read_parquet(f"{OUTPUT_DIR}/relationships.parquet")
communities_df = pd.read_parquet(f"{OUTPUT_DIR}/communities.parquet")

print(f"Loaded {len(entities_df)} entities, {len(relationships_df)} relationships, {len(communities_df)} communities")
print(f"Entity columns: {list(entities_df.columns)}")
print(f"Relationship columns: {list(relationships_df.columns)}")

In [ ]:
# Convert entities to yfiles format
def convert_entities_to_dicts(df, max_nodes=500):
    """Convert the entities dataframe to a list of dicts for yfiles-jupyter-graphs."""
    nodes_dict = {}
    count = 0
    
    for _, row in df.iterrows():
        if count >= max_nodes:
            break
            
        # Use 'id' or 'title' as node identifier
        node_id = row.get('id') or row.get('title') or row.get('name')
        
        if node_id and node_id not in nodes_dict:
            nodes_dict[node_id] = {
                "id": node_id,
                "properties": row.to_dict(),
            }
            count += 1
    
    return list(nodes_dict.values())

# Convert relationships to yfiles format
def convert_relationships_to_dicts(df, node_ids, max_edges=1000):
    """Convert the relationships dataframe to a list of dicts for yfiles-jupyter-graphs."""
    relationships = []
    count = 0
    
    for _, row in df.iterrows():
        if count >= max_edges:
            break
            
        source = row.get('source')
        target = row.get('target')
        
        # Only include edges where both nodes exist
        if source in node_ids and target in node_ids:
            relationships.append({
                "start": source,
                "end": target,
                "properties": row.to_dict(),
            })
            count += 1
    
    return relationships

In [ ]:
# Create interactive graph visualization
print("Creating interactive graph visualization...")

# Convert data
nodes = convert_entities_to_dicts(entities_df, max_nodes=200)
node_ids = {node["id"] for node in nodes}
edges = convert_relationships_to_dicts(relationships_df, node_ids, max_edges=500)

print(f"Visualizing {len(nodes)} nodes and {len(edges)} edges")

# Create widget
w = GraphWidget()
w.nodes = nodes
w.edges = edges
w.directed = True

# Configure appearance
w.node_label_mapping = lambda node: node["properties"].get("title") or node["properties"].get("name") or node["id"]

# Color by community if available
def community_to_color(community):
    colors = ["crimson", "darkorange", "indigo", "cornflowerblue", "cyan", "teal", "green", "purple", "pink", "brown"]
    return colors[int(community) % len(colors)] if community is not None else "lightgray"

if 'community' in entities_df.columns:
    w.node_color_mapping = lambda node: community_to_color(node["properties"].get("community"))

# Size by degree if available
if 'degree' in entities_df.columns:
    w.node_scale_factor_mapping = lambda node: 0.5 + (node["properties"].get("degree", 1) * 1.5 / 20)

# Edge thickness by weight
w.edge_thickness_factor_mapping = lambda edge: edge["properties"].get("weight", 1)

# Apply layout
w.circular_layout()  # Use circular layout for smaller graphs

# Display the widget
display(w)

## GraphML Export for Gephi

If GraphML snapshots are enabled in your settings.yaml, you can find .graphml files in the output directory.
These can be opened directly in Gephi for advanced visualization and analysis.

In [ ]:
# Check for GraphML files
import glob

graphml_files = glob.glob(f"{OUTPUT_DIR}/*.graphml")
if graphml_files:
    print(f"Found GraphML files: {graphml_files}")
    print("You can open these in Gephi for advanced visualization!")
else:
    print("No GraphML files found. Enable 'snapshots.graphml: true' in settings.yaml and re-run indexing.")